In [1]:
import pandas as pd
from scipy.stats import gmean

from yfetch import get_stock_history, get_stock_name
from symbols import my_etfs
    
symbols = my_etfs
rows = []
for symbol in symbols:
    history = get_stock_history(symbol, period='2y', interval='1d')
    monthly = history.resample('ME').agg({'High': 'max', 'Low': 'min'})
    monthly['Swing'] = 2 * (monthly.High - monthly.Low) / (monthly.High + monthly.Low)
    avg_swing = monthly.Swing.mean()
    last_price = history.Close.iloc[-1]
    one_year_ago = history.index[-1] - pd.DateOffset(years=1)
    past_year = history[history.index >= one_year_ago]
    prev_year = history[(history.index < one_year_ago)]
    avg_past_year = past_year.Close.mean()
    avg_prev_year = prev_year.Close.mean()
    yoy_change = avg_past_year / avg_prev_year - 1

    changes = history.Close.pct_change(periods=252, fill_method=None).dropna()
    gmean_change = gmean(1 + changes) - 1 # geometric mean of changes
    
    rows.append({
        'Symbol': symbol,
        'Name': get_stock_name(symbol),
        'YoY Avg': yoy_change,
        'G-Mean': gmean_change,
    })

results = pd.DataFrame(rows).set_index('Symbol')
results.to_csv('data/monthly-swing.csv')
for col in ['YoY Avg', 'G-Mean']:
    results[col] = results[col].map('{:.1%}'.format)
results

Fetched history for SPY (501 rows)
Fetched history for SPYG (501 rows)
Fetched history for QQQ (501 rows)
Fetched history for IGM (501 rows)
Fetched history for SPMO (501 rows)
Fetched history for MAGS (501 rows)
Fetched history for FNGS (501 rows)
Fetched history for QTUM (501 rows)
Fetched history for NUKZ (501 rows)
Fetched history for SMH (501 rows)
Fetched history for USD (501 rows)
Fetched history for PPA (501 rows)
Fetched history for DAPP (501 rows)
Fetched history for BITQ (501 rows)
Fetched history for ESIF.DE (505 rows)
Fetched history for VDIV.DE (503 rows)
Fetched history for JEDI.DE (503 rows)
Fetched history for IWMO.MI (501 rows)
Fetched history for XDWI.DE (503 rows)
Fetched history for 4GLD.DE (503 rows)
Fetched history for BTC-USD (731 rows)
Fetched history for GC=F (503 rows)


,Name,YoY Avg,G-Mean
Symbol,,,
SPY,State Street SPDR S&P 500 ETF Trust,20.1%,20.2%
SPYG,State Street SPDR Portfolio S&P 500 Growth ETF,25.7%,25.9%
QQQ,Invesco QQQ Trust,26.5%,26.6%
IGM,iShares Expanded Tech Sector ETF,35.1%,35.0%
SPMO,Invesco S&P 500 Momentum ETF,31.6%,31.5%
MAGS,Roundhill Magnificent Seven ETF,29.2%,29.8%
FNGS,MicroSectors FANG+ ETN,25.4%,25.9%
QTUM,Defiance Quantum ETF,61.4%,61.5%
NUKZ,Range Nuclear Renaissance ETF,54.8%,57.0%
